# Is the bottleneck data or stylometry? A CPU diagnostic for Name That LLM

Two opposite advice sets are on the table: "add more structural/stylometric signals (POS n-grams, compression, PLDA)" versus "the problem is topic leakage and data". Neither has been checked against your data. This notebook measures it, on CPU, in minutes to ~20 minutes, with **prompt-grouped** folds so topic cannot leak into validation.

**What it measures**
1. **Which views carry signal** (topic-aware: char, word; topic-blinded: shape, function-word skeleton, distortion with `*`; optional POS; a compression-distance baseline that tests the NCD claim).
2. **Learning curve** on training size (and on added external rows). A steep slope at full data means more data is the lever. A flat one means the sparse members are saturated.
3. **Error structure.** Are mistakes between *sibling* models (same family) or across families? Sibling errors call for fine-grained, family-specific work, not a new generic feature.
4. **Leakage test** for external data: the same model with and without external rows that nearly duplicate validation passages. This is the cheapest explanation of "OOF high, leaderboard low".
5. **Your unseen set** (optional): out-of-sample accuracy plus an exported probability file you can plug into the inference notebook as `OUR_MEMBER_PROBS`.

**What it cannot tell you.** It evaluates sparse, CPU-only members. It says whether *those* are data-bound or stylometry-bound, and whether a structural member is worth adding. It does not evaluate the LoRA models.

If `train.csv` is not found the notebook runs on synthetic data (smoke mode) only to check that every cell works.

In [ ]:
import os, re, gc, glob, json, time, zlib, warnings
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from scipy.sparse import hstack
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupKFold
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, accuracy_score, confusion_matrix
warnings.filterwarnings('ignore')

CFG = dict(
    n_folds=5, n_eval_folds=5,         # lower n_eval_folds (e.g. 3) for a faster, noisier run
    feature_scale=1.0,                 # multiply max_features (0.5 for a faster run)
    fractions=[0.1, 0.25, 0.5, 0.75, 1.0],
    extra_paths=[],                    # optional labelled extras (TRAIN ONLY), e.g. ['/kaggle/input/xxx/gen.csv']
    extra_cap=4000, extra_weight=0.5,          # total extra rows
    extra_cap_per_label=4000, extra_cap_source_share=0.30,
    extra_sample_by_match_weight=False,
    imbalance_seeds=[0, 1],
    unseen_path=None,                  # your generated labeled set: text, model (+ optional id, prompt_id)
    ncd_rows=300, ncd_block_bytes=28000,
    seed=0,
)
SEED = CFG['seed']; rs = np.random.RandomState(SEED)
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'; FIG = os.path.join(OUT, 'figs_diag'); os.makedirs(FIG, exist_ok=True)
def find(name):
    pref = f'/kaggle/input/competitions/name-that-llm/{name}'      # several attached datasets may also contain a train.csv; prefer the competition's
    if os.path.exists(pref): return pref
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True); return hits[0] if hits else None
SMOKE = find('train.csv') is None
LABELS_18 = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5','gemini-3.1-pro-preview','gemini-3.6-flash',
             'gemini-3.7-flash','gemini-3.8-flash','glm-5.3','glm-5.3-flash','gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','kimi-k3','qwen3.8-max']
FAMILY = lambda l: re.match(r'[a-z]+', l).group(0)
print('SMOKE MODE (synthetic data)' if SMOKE else 'REAL MODE')

## 1. Data and prompt-grouped folds

In [ ]:
def lab_norm(s): return re.sub(r'[\s_]+', '-', str(s).strip().lower())
LABEL_MAP = {lab_norm(l): l for l in LABELS_18}; LABEL_MAP.update({lab_norm(l).replace('.', '-'): l for l in LABELS_18})
def read_any(p): return pd.read_parquet(p) if p.endswith('.parquet') else pd.read_json(p, lines=True) if p.endswith('.jsonl') else pd.read_csv(p)
def load_labeled(path):
    raw = read_any(path)
    tc = next(c for c in ('text', 'passage', 'generation', 'response') if c in raw.columns); lc = next(c for c in ('model', 'label', 'llm', 'generator') if c in raw.columns)
    df = pd.DataFrame({'text': raw[tc].astype(str), 'model_raw': raw[lc].astype(str)})
    if 'source_dataset' in raw.columns:
        df['source'] = raw['source_dataset'].astype(str)
    if 'prompt_canon' in raw.columns:
        pcv = raw['prompt_canon'].fillna('').astype(str)
        df['prompt_id'] = pcv.where(pcv.str.startswith('p'), df.get('source', '').astype(str) + ':' + pcv)
        df.loc[pcv.eq(''), 'prompt_id'] = np.nan
    elif 'prompt_id' in raw.columns:
        df['prompt_id'] = raw['prompt_id'].astype(str)
    for col in ('match_weight', 'panel_labels', 'prompt_evidence', 'flag_kb_proxy', 'n_sources'):
        if col in raw.columns: df[col] = raw[col].values
    df['model'] = df.model_raw.map(lambda s: LABEL_MAP.get(lab_norm(s)) or LABEL_MAP.get(lab_norm(s).replace('.', '-')))
    df['id'] = raw['id'].astype(str).values if 'id' in raw.columns else [f'ev{i:06d}' for i in range(len(raw))]     # same id scheme as the inference notebook
    if 'prompt_id' not in df: df['prompt_id'] = np.nan
    bad = df[df.model.isna()].model_raw.value_counts()
    if len(bad): print(f'  {os.path.basename(path)}: unknown labels dropped', bad.head(6).to_dict())
    return df.dropna(subset=['model']).query('text.str.len() > 50', engine='python').reset_index(drop=True)

if SMOKE:
    nm = lambda pre, i: pre + chr(97 + i // 26) + chr(97 + i % 26)
    pool = [nm('top', i) for i in range(500)]; fill = [nm('fil', i) for i in range(80)]
    fam_words = {f: rs.choice(fill, 6, replace=False) for f in {FAMILY(l) for l in LABELS_18}}
    own = {l: rs.choice(fill, 4, replace=False) for l in LABELS_18}
    def synth(tag, n_prompts):
        rows = []
        for p in range(n_prompts):
            topic = rs.choice(pool, 14, replace=False)
            for l in LABELS_18:
                w = [rs.choice(topic) if rs.rand() < .42 else rs.choice(fam_words[FAMILY(l)]) if rs.rand() < .4 else rs.choice(own[l]) if rs.rand() < .25 else rs.choice(fill) for _ in range(80)]
                rows.append((f'{tag}{p}_{l}', f'{tag}{p}', ' '.join(w) + '.', l))
        return pd.DataFrame(rows, columns=['id', 'prompt_id', 'text', 'model'])
    train = synth('p', 30); unseen = synth('u', 6); unseen['model_raw'] = unseen.model
else:
    train = pd.read_csv(find('train.csv')); unseen = None
LABELS = sorted(train.model.unique()); K = len(LABELS); L2I = {l: i for i, l in enumerate(LABELS)}
fam = np.array([FAMILY(l) for l in LABELS])
train = train.reset_index(drop=True); train['y'] = train.model.map(L2I)
pcode = pd.factorize(train.prompt_id)[0]
fold = np.full(len(train), -1)
for f, (_, va) in enumerate(GroupKFold(CFG['n_folds']).split(np.arange(len(train)), groups=pcode)): fold[va] = f
FOLDS = list(range(min(CFG['n_eval_folds'], CFG['n_folds'])))

extra = pd.DataFrame(columns=['text', 'model', 'y'])
if CFG['extra_paths']:
    nt = lambda t: re.sub(r'\s+', ' ', str(t)).strip().lower(); seen = set(train.text.map(nt))
    ex = pd.concat([load_labeled(p) for p in CFG['extra_paths']]).drop_duplicates('text'); ex = ex[~ex.text.map(nt).isin(seen)]
    cap = max(1, min(CFG['extra_cap_per_label'], CFG['extra_cap'] // len(LABELS_18)))
    chunks = []
    for _, d in ex.groupby('model', sort=False):
        if 'source' in d.columns and CFG['extra_cap_source_share'] < 1:
            per_source = max(1, int(cap * CFG['extra_cap_source_share']))
            d = pd.concat([s.sample(min(len(s), per_source), random_state=SEED) for _, s in d.groupby('source', sort=False)])
        if CFG['extra_sample_by_match_weight'] and 'match_weight' in d.columns:
            d = d.sample(min(len(d), cap), weights=d['match_weight'].clip(lower=0), random_state=SEED)
        else:
            d = d.sample(min(len(d), cap), random_state=SEED)
        chunks.append(d)
    extra = pd.concat(chunks, ignore_index=True).reset_index(drop=True); extra['y'] = extra.model.map(L2I)
    print('extra rows (train only):', len(extra), extra.model.value_counts().min(), '-', extra.model.value_counts().max(), 'per class')
print(f'train {len(train)} rows, {train.prompt_id.nunique()} prompts, {K} classes | folds {FOLDS} | extras {len(extra)}')
fig, ax = plt.subplots(figsize=(11, 3.4)); order = train.assign(w=train.text.str.split().str.len()).groupby('model').w.median().sort_values().index
ax.boxplot([train[train.model == m].text.str.split().str.len() for m in order], labels=order, showfliers=False); plt.setp(ax.get_xticklabels(), rotation=60, ha='right', fontsize=7)
ax.set_title('Passage length per model'); ax.set_ylabel('words'); plt.tight_layout(); plt.savefig(f'{FIG}/00_lengths.png', dpi=110); plt.show()

## 2. Views: topic-aware and topic-blinded

In [ ]:
FUNCTION_WORDS = frozenset('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
def shape_view(t): return re.sub(r'[^\W\d_]+', lambda m: 'X' if m[0].isupper() else 'x', t)
def function_view(t): return ' '.join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else 'CONTENT' for w in re.findall(r'\w+|[^\w\s]', t))
def distort_view(t): return re.sub(r"[A-Za-z\u2019']+", lambda m: m[0] if m[0].lower() in FUNCTION_WORDS else '*', t)
S = CFG['feature_scale']
VIEWS = {   # name: (preprocess, analyzer, ngram, max_features, weight, topic_blinded)
    'char':     (None,          'char', (3, 5), int(250000 * S), 1.0, False),
    'word':     (None,          'word', (1, 3), int(180000 * S), 0.7, False),
    'shape':    (shape_view,    'char', (2, 6), int(60000 * S),  0.5, True),
    'function': (function_view, 'word', (1, 3), int(50000 * S),  0.5, True),
    'distort':  (distort_view,  'char', (3, 5), int(100000 * S), 0.5, True),
}
try:
    import spacy
    NLP = spacy.load('en_core_web_sm', disable=['ner', 'lemmatizer', 'parser'])
    pos_view = lambda t: ' '.join(tok.pos_ for tok in NLP(t[:3000]))
    VIEWS['pos'] = (pos_view, 'word', (2, 4), 50000, 0.5, True); print('POS view enabled (spaCy tagger found)')
except Exception:
    print('POS view skipped: no offline spaCy tagger. Attach a spaCy model wheel to test it; the other blinded views are the proxy.')

def build_view(name, fit_texts, *transform_sets):
    prep, an, ng, mf, w, _ = VIEWS[name]; P = (lambda T: [prep(t) for t in T]) if prep else (lambda T: list(T))
    opts = dict(analyzer=an, ngram_range=ng, lowercase=False, sublinear_tf=True, min_df=2, max_features=mf, dtype=np.float32)
    if an == 'word' and name == 'function': opts['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]'
    elif an == 'word': opts['token_pattern'] = r'(?u)\b\w+\b|[^\w\s]' if name == 'word' else r'\S+'
    v = TfidfVectorizer(**opts).fit(P(fit_texts)); return [v.transform(P(T)) * w for T in transform_sets]

FM = {}   # fold -> {view: (Xtr, Xva, Xextra)}
def fold_views(f, names):
    tr, va = np.where(fold != f)[0], np.where(fold == f)[0]; got = FM.setdefault(f, {})
    for n in names:
        if n in got: continue
        t0 = time.time(); sets = [train.text.values[tr], train.text.values[va]] + ([extra.text.values] if len(extra) else [])
        out = build_view(n, train.text.values[tr], *sets); got[n] = (out[0], out[1], out[2] if len(extra) else None)
    return tr, va

def softmax(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def fit_predict(f, names, frac=1.0, extra_frac=0.0, extra_mask=None, seed=0):
    tr, va = fold_views(f, names); r = np.random.RandomState(seed)
    if frac < 1.0:      # subsample whole prompts so groups stay intact
        pr = np.unique(pcode[tr]); keep = set(r.choice(pr, max(2, int(round(frac * len(pr)))), replace=False)); sel = np.array([i for i, t in enumerate(tr) if pcode[t] in keep])
    else: sel = np.arange(len(tr))
    Xtr = hstack([FM[f][n][0] for n in names], format='csr')[sel]; y = train.y.values[tr][sel]; w = np.ones(len(sel))
    if extra_frac > 0 and len(extra):
        ei = np.arange(len(extra)) if extra_mask is None else np.where(extra_mask)[0]
        ei = r.choice(ei, int(len(ei) * extra_frac), replace=False) if extra_frac < 1 else ei
        Xe = hstack([FM[f][n][2] for n in names], format='csr')[ei]
        from scipy.sparse import vstack
        Xtr = vstack([Xtr, Xe], format='csr'); y = np.r_[y, extra.y.values[ei]]; w = np.r_[w, np.full(len(ei), CFG['extra_weight'])]
    clf = LinearSVC(C=0.5, class_weight='balanced', max_iter=3000, random_state=SEED).fit(Xtr, y, sample_weight=w)
    D = np.full((len(va), K), -5.0); D[:, clf.classes_] = clf.decision_function(hstack([FM[f][n][1] for n in names], format='csr')); return va, D, Xtr.shape[0]

def score_set(names, **kw):
    accs, bals, oof = [], [], np.zeros((len(train), K)); ns = []
    for f in FOLDS:
        va, D, n = fit_predict(f, names, **kw); oof[va] = D; ns.append(n)
        accs.append(accuracy_score(train.y.values[va], D.argmax(1))); bals.append(balanced_accuracy_score(train.y.values[va], D.argmax(1)))
    return dict(acc=np.mean(accs), acc_sd=np.std(accs), bal=np.mean(bals), n_train=float(np.mean(ns))), oof

In [ ]:
t0 = time.time(); R = {}; OOF = {}
single = [n for n in VIEWS]
for n in single: R[n], OOF[n] = score_set([n]); print(f'{n:9s} alone: acc {R[n]["acc"]:.3f} +- {R[n]["acc_sd"]:.3f}  bal {R[n]["bal"]:.3f}  ({time.time()-t0:.0f}s)', flush=True)
ref4 = ['char', 'word', 'shape', 'function']; blind = [n for n in VIEWS if VIEWS[n][5]]
R['reference SVC4 (char+word+shape+function)'], OOF['ref4'] = score_set(ref4)
R['all views'], OOF['all'] = score_set(list(VIEWS))
R['topic-blinded views only'], OOF['blind'] = score_set(blind)
for k in list(R)[len(single):]: print(f'{k:42s}: acc {R[k]["acc"]:.3f} +- {R[k]["acc_sd"]:.3f}  bal {R[k]["bal"]:.3f}', flush=True)
print(f'views done in {time.time()-t0:.0f}s')

In [ ]:
# compression-distance baseline (tests the NCD claim): cost of the passage given a class dictionary of ~28 KB of that class's training text
def ncd_eval(f, n_rows):
    tr, va = np.where(fold != f)[0], np.where(fold == f)[0]; va = va[:n_rows] if len(va) > n_rows else va; dicts = []
    for k in range(K):
        idx = tr[train.y.values[tr] == k]; r = np.random.RandomState(k).permutation(idx); buf = b''
        for i in r:
            if len(buf) >= CFG['ncd_block_bytes']: break
            buf += train.text.values[i].encode()[:4000] + b'\n'
        dicts.append(buf[-32768:])
    cost = np.zeros((len(va), K))
    for j, i in enumerate(va):
        x = train.text.values[i].encode()
        for k in range(K):
            c = zlib.compressobj(9, zlib.DEFLATED, 15, 9, zlib.Z_DEFAULT_STRATEGY, dicts[k]); cost[j, k] = len(c.compress(x) + c.flush())
    return balanced_accuracy_score(train.y.values[va], cost.argmin(1)), len(va)
t0 = time.time(); a, nn_ = ncd_eval(FOLDS[0], CFG['ncd_rows']); R['compression distance (zlib dictionary)'] = dict(acc=a, acc_sd=0, bal=a, n_train=0)
print(f'compression-distance accuracy on {nn_} validation passages: {a:.3f} (chance {1/K:.3f}), {time.time()-t0:.0f}s')

names = list(R); vals = [R[k]['acc'] for k in names]; sds = [R[k]['acc_sd'] for k in names]
col = ['tab:green' if (k in VIEWS and VIEWS[k][5]) or k.startswith('topic-blinded') else 'tab:gray' if k.startswith('compression') else 'tab:blue' for k in names]
fig, ax = plt.subplots(figsize=(10, 0.42 * len(names) + 1.4)); ax.barh(range(len(names)), vals, xerr=sds, color=col, capsize=2); ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
ax.invert_yaxis(); ax.axvline(1 / K, color='k', ls=':'); ax.set_xlim(0, max(vals) * 1.15); ax.set_xlabel('accuracy on unseen prompts (grouped CV, mean +- sd over folds)')
for i, v in enumerate(vals): ax.text(v + .004, i, f'{v:.3f}', va='center', fontsize=8)
ax.set_title('Which signal survives unseen prompts? (green = topic-blinded, blue = topic-aware, gray = compression)'); plt.tight_layout(); plt.savefig(f'{FIG}/01_views.png', dpi=110); plt.show()

## 3. Learning curve: is it data-bound?

In [ ]:
t0 = time.time(); LC = []
for fr in CFG['fractions']:
    res, _ = score_set(ref4, frac=fr); LC.append((res['n_train'], res['acc'], res['acc_sd'], fr)); print(f'  {fr:>5.2f} of the training prompts -> {res["n_train"]:.0f} rows: acc {res["acc"]:.3f} +- {res["acc_sd"]:.3f}', flush=True)
LCe = []
if len(extra):
    for ef in (0.25, 0.5, 1.0):
        res, _ = score_set(ref4, extra_frac=ef); LCe.append((res['n_train'], res['acc'], res['acc_sd'], ef)); print(f'  full train + {ef:.0%} of extras -> {res["n_train"]:.0f} rows: acc {res["acc"]:.3f}', flush=True)
fig, ax = plt.subplots(figsize=(8, 4.8)); x, y, s = zip(*[(a, b, c) for a, b, c, _ in LC])
ax.errorbar(x, y, yerr=s, fmt='o-', label='train prompts subsampled'); 
if LCe: ax.errorbar([a for a, *_ in LCe], [b for _, b, *_ in LCe], yerr=[c for *_, c, _ in LCe], fmt='s--', color='tab:orange', label='full train + external rows')
ax.set_xscale('log'); ax.set_xlabel('training rows (log)'); ax.set_ylabel('accuracy on unseen prompts'); ax.legend(); ax.set_title('Learning curve of the sparse SVC4 member'); plt.tight_layout(); plt.savefig(f'{FIG}/02_learning_curve.png', dpi=110); plt.show()
half = [r for r in LC if abs(r[3] - 0.5) < 1e-9]; fullr = [r for r in LC if abs(r[3] - 1.0) < 1e-9]
GAIN_DOUBLING = (fullr[0][1] - half[0][1]) * 100 if half and fullr else float('nan')
print(f'gain from doubling the training data (50% -> 100%): {GAIN_DOUBLING:+.1f} points; learning-curve time {time.time()-t0:.0f}s')

## 4. Error structure: siblings or whole families?

In [ ]:
D = OOF['ref4']; mask = np.isin(np.arange(len(train)), np.concatenate([np.where(fold == f)[0] for f in FOLDS]))
yt, yp = train.y.values[mask], D[mask].argmax(1); err = yt != yp
same_fam = fam[yt] == fam[yp]; within = float((err & same_fam).sum() / max(err.sum(), 1))
print(f'errors that stay inside the true family (siblings): {within:.1%} of {int(err.sum())} errors')
order = np.argsort(fam, kind='stable'); cm = confusion_matrix(yt, yp, labels=np.arange(K)); cmn = (cm / np.maximum(cm.sum(1, keepdims=True), 1))[np.ix_(order, order)]
fig, ax = plt.subplots(1, 2, figsize=(16, 6.4), gridspec_kw=dict(width_ratios=[1.3, 1])); im = ax[0].imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax[0].set_xticks(range(K)); ax[0].set_yticks(range(K)); ax[0].set_xticklabels(np.array(LABELS)[order], rotation=80, ha='right', fontsize=6); ax[0].set_yticklabels(np.array(LABELS)[order], fontsize=6)
b = np.where(np.diff(pd.factorize(fam[order])[0]) != 0)[0] + .5
for v in b: ax[0].axhline(v, color='red', lw=.7); ax[0].axvline(v, color='red', lw=.7)
ax[0].set_title('SVC4 confusion (red lines separate families)')
pf = pd.DataFrame({'fam': fam[yt], 'ok': ~err}).groupby('fam').ok.mean().sort_values(); ax[1].barh(pf.index, pf.values, color='tab:blue'); ax[1].set_xlim(0, 1); ax[1].set_title('Accuracy per family')
plt.tight_layout(); plt.savefig(f'{FIG}/03_family_confusion.png', dpi=110); plt.show()
pairs = pd.DataFrame([(LABELS[i], LABELS[j], cm[i, j]) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0], columns=['true', 'predicted', 'count']).sort_values('count', ascending=False).head(8)
print(pairs.to_string(index=False))

## 5. Leakage test for external data

In [ ]:
LEAK = None
if len(extra):
    from sklearn.feature_extraction.text import TfidfVectorizer as TV
    v = TV(stop_words='english', sublinear_tf=True, min_df=2).fit(list(train.text) + list(extra.text)); E = v.transform(extra.text); res = []
    for f in FOLDS:
        va = np.where(fold == f)[0]; sim = np.asarray((E @ v.transform(train.text.values[va]).T).max(axis=1).todense()).ravel(); res.append(sim)
    sim_max = np.max(res, axis=0)   # worst case over the validation folds
    print(f'extra rows whose nearest validation passage has cosine >0.5: {np.mean(sim_max > .5):.1%}, >0.7: {np.mean(sim_max > .7):.1%}')
    accs = {}
    for tag, keep in (('all extras', None), ('without near-duplicates of validation rows', np.array([float(np.max([r[i] for r in res])) <= 0.5 for i in range(len(extra))]))):
        a, _ = score_set(ref4, extra_frac=1.0, extra_mask=keep); accs[tag] = a['acc']; print(f'  {tag:44s}: acc {a["acc"]:.3f}')
    LEAK = accs['all extras'] - accs['without near-duplicates of validation rows']
    print(f'accuracy inflation from near-duplicate extras: {LEAK*100:+.1f} points (positive = your OOF is flattered by external data)')
else:
    print('No extras supplied (CFG["extra_paths"]); this test is skipped.')

## 6. Your unseen set (optional) and a probability file for the inference notebook

In [ ]:
UNSEEN_ACC = None
if SMOKE or CFG['unseen_path']:
    UN = unseen.assign(id=unseen.id.astype(str)).reset_index(drop=True) if SMOKE else load_labeled(CFG['unseen_path'])
    nt = lambda t: re.sub(r'\s+', ' ', str(t)).strip().lower(); UN = UN[~UN.text.map(nt).isin(set(train.text.map(nt)))].reset_index(drop=True)
    names = ref4; mats_tr = build_view; tr_all = np.arange(len(train)); sets = [train.text.values, UN.text.values] + ([extra.text.values] if len(extra) else [])
    Xs = [build_view(n, train.text.values, *sets) for n in names]
    Xtr = hstack([x[0] for x in Xs], format='csr'); Xun = hstack([x[1] for x in Xs], format='csr'); y = train.y.values; w = np.ones(len(train))
    if len(extra):
        from scipy.sparse import vstack
        Xtr = vstack([Xtr, hstack([x[2] for x in Xs], format='csr')], format='csr'); y = np.r_[y, extra.y.values]; w = np.r_[w, np.full(len(extra), CFG['extra_weight'])]
    clf = LinearSVC(C=0.5, class_weight='balanced', max_iter=3000, random_state=SEED).fit(Xtr, y, sample_weight=w)
    Dun = np.full((len(UN), K), -5.0); Dun[:, clf.classes_] = clf.decision_function(Xun); yu = UN.model.map(L2I).values
    UNSEEN_BAL = balanced_accuracy_score(yu, Dun.argmax(1)); print(f'unseen set balanced accuracy {UNSEEN_BAL:.3f}; rows {len(UN)}')
    P = softmax(3 * Dun); outdf = pd.DataFrame(P, columns=LABELS); outdf.insert(0, 'id', UN.id.values); outdf.to_csv(f'{OUT}/structural_member_probs.csv', index=False)
    print('wrote structural_member_probs.csv: set OUR_MEMBER_PROBS to it in the inference notebook (aligned by id).')
else:
    print('No unseen set (CFG["unseen_path"]).')

## 7. Verdict

In [ ]:
best_blind = max((k for k in R if k in VIEWS and VIEWS[k][5]), key=lambda k: R[k]['acc']); best_aware = max(('char', 'word'), key=lambda k: R[k]['acc'])
print(f'1. Topic-aware best ({best_aware}) {R[best_aware]["acc"]:.3f} vs topic-blinded best ({best_blind}) {R[best_blind]["acc"]:.3f} vs all blinded together {R["topic-blinded views only"]["acc"]:.3f}')
print('   ->', 'blinded views keep most of the signal: structure matters and topic is not the whole story' if R['topic-blinded views only']['acc'] >= .85 * R['all views']['acc'] else
      'blinded views lose a lot, the topic-aware n-grams are carrying real style information on unseen prompts')
print(f'2. Reference SVC4 {R["reference SVC4 (char+word+shape+function)"]["acc"]:.3f} -> all views {R["all views"]["acc"]:.3f}: adding more views changes accuracy by {100*(R["all views"]["acc"]-R["reference SVC4 (char+word+shape+function)"]["acc"]):+.1f} points')
print(f'   compression distance {R["compression distance (zlib dictionary)"]["acc"]:.3f} (chance {1/K:.3f}): ', 'not competitive, skip NCD' if R['compression distance (zlib dictionary)']['acc'] < .6 * R['reference SVC4 (char+word+shape+function)']['acc'] else 'worth a look')
print(f'3. Doubling the training data (50% -> 100%) gains {GAIN_DOUBLING:+.1f} points:', 'DATA-BOUND (steep)' if GAIN_DOUBLING >= 2 else 'sparse members are saturating, more of the same data helps little' if GAIN_DOUBLING == GAIN_DOUBLING else 'n/a')
print(f'4. {within:.0%} of errors are inside a family:', 'sibling discrimination is the problem: family-specific heads and targeted data for those classes' if within >= .5 else 'errors cross families: a representation (LoRA) or data-distribution problem')
if LEAK is not None: print(f'5. External-data leakage: {LEAK*100:+.1f} points', '(large: your OOF is not trustworthy, filter near-duplicates and regroup)' if LEAK > .02 else '(small)')
if UNSEEN_ACC is not None: print(f'6. Unseen set {UNSEEN_ACC:.3f} vs grouped CV {R["reference SVC4 (char+word+shape+function)"]["acc"]:.3f}: gap {100*(UNSEEN_ACC-R["reference SVC4 (char+word+shape+function)"]["acc"]):+.1f} points')

In [ ]:
# v12 imbalance experiment: compare natural and capped/match-weighted extra sampling.
if not SMOKE and len(extra):
    imbalance_rows = []
    for seed in CFG['imbalance_seeds']:
        for plan in ('natural', 'capped', 'capped_weighted'):
            rng = np.random.RandomState(seed)
            sample = extra if plan == 'natural' else pd.concat([d.sample(min(len(d), CFG['extra_cap_per_label']), random_state=seed) for _, d in extra.groupby('model', sort=False)], ignore_index=True)
            imbalance_rows.append({'seed': seed, 'plan': plan, 'rows': len(sample)})
    print('imbalance experiment prepared:', pd.DataFrame(imbalance_rows).groupby('plan').rows.agg(['mean','min','max']).to_dict())
